In [1]:
# Тема 4. Гіперпараметри моделей машинного навчання та методи їх оптимізації

**Виконав Гуленко Назар, студент групи ІДС-501**

In [2]:
import pandas as pd
df = pd.read_csv('digital.csv')
df.head()

,user_id,age,gender,occupation,work_mode,screen_time_hours,work_screen_hours,leisure_screen_hours,sleep_hours,sleep_quality_1_5,stress_level_0_10,productivity_0_100,exercise_minutes_per_week,social_hours_per_week,mental_wellness_index_0_100,Unnamed: 15
0,U0001,33,Female,Employed,Remote,10.79,5.44,5.35,6.63,1,9.3,44.7,127,0.7,9.3,NaN
1,U0002,28,Female,Employed,In-person,7.40,0.37,7.03,8.05,3,5.7,78.0,74,2.1,56.2,NaN
2,U0003,35,Female,Employed,Hybrid,9.78,1.09,8.69,6.48,1,9.1,51.8,67,8.0,3.6,NaN
3,U0004,42,Male,Employed,Hybrid,11.13,0.56,10.57,6.89,1,10.0,37.0,0,5.7,0.0,NaN
4,U0005,28,Male,Student,Remote,13.22,4.09,9.13,5.79,1,10.0,38.5,143,10.1,0.0,NaN


In [3]:
df.info()
print(df.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 16 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   user_id                      400 non-null    object 
 1   age                          400 non-null    int64  
 2   gender                       400 non-null    object 
 3   occupation                   400 non-null    object 
 4   work_mode                    400 non-null    object 
 5   screen_time_hours            400 non-null    float64
 6   work_screen_hours            400 non-null    float64
 7   leisure_screen_hours         400 non-null    float64
 8   sleep_hours                  400 non-null    float64
 9   sleep_quality_1_5            400 non-null    int64  
 10  stress_level_0_10            400 non-null    float64
 11  productivity_0_100           400 non-null    float64
 12  exercise_minutes_per_week    400 non-null    int64  
 13  social_hours_per_wee

Бачимо що у наборі даних є колонка Unnamed: 15, в якій немає значень, тому її приберемо для зручночті роботи з даними.

In [4]:
df = df.drop(columns=['Unnamed: 15'])

**1.Підготовка цільової змінної та даних**


Проведемо бінаризацію цільової змінної (ментального благополуччя - mental_wellness_index_0_100)

In [5]:
# медіана
median_value = df['mental_wellness_index_0_100'].median()

# створюємо бінарну змінну
df['target'] = (df['mental_wellness_index_0_100'] > median_value).astype(int)

df['target'].value_counts()

target
0    201
1    199
Name: count, dtype: int64

In [6]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# видаляємо зайве
X = df.drop(columns=['user_id', 'mental_wellness_index_0_100', 'target'])
y = df['target']

# one-hot encoding
X = pd.get_dummies(X, drop_first=True)

# train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=123, stratify=y
)

# стандартизація
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**2.Побудова моделі логістичної регресії**

Оскільки ми розділили дані на дві рівні частини за медіаною, у вибірці немає перекосу в бік одного з класів. Це дозволяє нам покладатися на Accuracy, оскільки за таких умов метрика не буде викривляти реальну якість роботи моделі

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

model = LogisticRegression(max_iter=5000)

param_grid = {
    "C": [0.01, 0.1, 1, 10, 100]
}

grid = GridSearchCV(
    model,
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid.fit(X_train_scaled, y_train)

print("Best C:", grid.best_params_)
print("Best CV score:", grid.best_score_)

test_score_log = grid.score(X_test_scaled, y_test)
print("Test accuracy:", test_score_log)

Best C: {'C': 10}
Best CV score: 0.9142857142857144
Test accuracy: 0.8916666666666667


Оцінка моделі проводилася за двоетапною схемою. Спочатку методом крос-валідації ($k=5$) аналізувалася точність для кожного варіанту параметра $C$. Потім модель з обраним оптимальним гіперпараметром була протестована на даних, що не використовувалися під час навчання. 

Досягнута точність у 0.892 свідчить про ефективне узагальнення закономірностей. Схожість результатів CV score та тестової метрики доводить стійкість моделі до оверфіттингу

**3.Дерево рішень з обмеженням глибини**

In [8]:
from sklearn.tree import DecisionTreeClassifier

param_grid = {
    "max_depth": [2, 3, 5, 7, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5]
}

grid_tree = GridSearchCV(
    DecisionTreeClassifier(random_state=123),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid_tree.fit(X_train, y_train)

print("Best params:", grid_tree.best_params_)
print("Best CV:", grid_tree.best_score_)

test_score_tree = grid_tree.score(X_test, y_test)
print("Test accuracy:", test_score_tree)

Best params: {'max_depth': 3, 'min_samples_leaf': 5, 'min_samples_split': 2}
Best CV: 0.9285714285714285
Test accuracy: 0.8583333333333333


Використання дерева з максимальною глибиною 3 дозволило досягти показника CV accuracy 0.929, проте на незалежному тестовому наборі точність впала до 0.858. Така розбіжність вказує на певний рівень перенавчання (overfitting). Отже, обмеження глибини не повністю нівелювало підлаштування моделі під особливості тренувальної вибірки, що негативно вплинуло на її здатність до узагальнення на нових спостереженнях

**4.Дерево рішень з обрізанням (pruning)**

In [9]:
full_tree = DecisionTreeClassifier(random_state=42)
full_tree.fit(X_train, y_train)

path = full_tree.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[:-1]

param_grid = {"ccp_alpha": ccp_alphas}

grid_prune = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="accuracy"
)

grid_prune.fit(X_train, y_train)

print("Best alpha:", grid_prune.best_params_)
print("Best CV:", grid_prune.best_score_)

test_score_prune = grid_prune.score(X_test, y_test)
print("Test accuracy:", test_score_prune)

Best alpha: {'ccp_alpha': np.float64(0.013486167800453511)}
Best CV: 0.9178571428571429
Test accuracy: 0.8666666666666667


Застосування методу обрізання дозволило збалансувати складність моделі, зробивши її менш чутливою до шумів у навчальних даних. Показник тестової точності зріс до 0.867, попри легке просідання CV-метрики (0.918). Це демонструє класичний ефект регуляризації: шляхом навмисного спрощення структури дерева ми зменшуємо перенавчання та отримуємо модель, яка демонструє стабільніші результати при переході від навчання до тестування

**5.Випадковий ліс**

Зважаючи на складність архітектури випадкового лісу, ми застосовуємо RandomSearch як ефективніший інструмент для пошуку в багатовимірному просторі параметрів. Разом з тим, з метою верифікації отриманих значень та порівняння точності обох методів, було вирішено додати етап оптимізації через GridSearch

**RandomSearchCV**

In [10]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5]
}

rf = RandomForestClassifier(random_state=42)

random_search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist,
    n_iter=10,
    cv=5,
    scoring="accuracy",
    random_state=42
)

random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)
print("Best CV:", random_search.best_score_)

test_score_rf = random_search.score(X_test, y_test)
print("Test accuracy:", test_score_rf)

Best params: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_depth': 7}
Best CV: 0.9071428571428573
Test accuracy: 0.8666666666666667


Результати налаштування випадкового лісу вказують на те, що ансамблевий підхід не став визначальним фактором успіху для цього набору даних. Різниця в якості між складним лісом та окремим деревом з обрізанням залишається незначною. Це дозволяє зробити висновок про надмірність високої складності моделі для поточної структури даних, де простіші алгоритми демонструють аналогічну ефективність

**GridSearchCV**

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
import time

# модель
rf = RandomForestClassifier(random_state=42)

# той самий простір параметрів (але тепер повний перебір)
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [3, 5, 7, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 5]
}

start_time = time.time()

grid_search_rf = GridSearchCV(
    rf,
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search_rf.fit(X_train, y_train)

end_time = time.time()

print("Best params:", grid_search_rf.best_params_)
print("Best CV:", grid_search_rf.best_score_)

test_score_rf_grid = grid_search_rf.score(X_test, y_test)
print("Test accuracy:", test_score_rf_grid)

print(f"Час Grid Search: {end_time - start_time:.2f} секунд")

Best params: {'max_depth': 7, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Best CV: 0.9107142857142858
Test accuracy: 0.8833333333333333
Час Grid Search: 53.85 секунд


Вичерпний пошук за сіткою продемонстрував найкращу предиктивну здатність серед моделей на основі дерев, продемонструвавши перевагу детального налаштування над Randomized Search. Тим не менш, такий підхід продемонстрував низьку ефективність щодо співвідношення результату до витрачених ресурсів: виграш у точності є маргінальним порівняно зі значним обчислювальним навантаженням

**6.Порівняння моделей та узагальнення результатів**

In [12]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Decision Tree", "Pruned Tree", "Random Forest (RandomizedSearchCV)","Random Forest (GridSearchCV)"],
    "Test Accuracy": [test_score_log, test_score_tree, test_score_prune, test_score_rf, test_score_rf_grid]
})

print(results)

                                Model  Test Accuracy
0                 Logistic Regression       0.891667
1                       Decision Tree       0.858333
2                         Pruned Tree       0.866667
3  Random Forest (RandomizedSearchCV)       0.866667
4        Random Forest (GridSearchCV)       0.883333


Лідером за метриками на етапі тестування стала логістична регресія. 
Судячи з отриманих даних, використання складних архітектур є малоефективним: вони не дають відчутного виграшу в якості, але потребують значно більше часу на навчання. Це дозволяє зробити висновок, що характер залежностей у даних є близьким до лінійного. Хоча невеликі розбіжності в результатах і вказують на наявність певних нелінійних ефектів, їхній вплив на підсумковий результат прогнозування залишається несуттєвим

**7.Інтерпретація результатів**

In [13]:
import numpy as np

best_model = grid.best_estimator_

coefficients = pd.DataFrame({
    "feature": X.columns,
    "coef": best_model.coef_[0]
}).sort_values(by="coef", key=abs, ascending=False)

print(coefficients.head(10))

                      feature      coef
6           stress_level_0_10 -4.648568
5           sleep_quality_1_5  2.632292
7          productivity_0_100  1.466906
14         occupation_Student -0.983500
17           work_mode_Remote  0.764189
11    gender_Non-binary/Other  0.598076
16        work_mode_In-person  0.580432
2           work_screen_hours -0.579286
8   exercise_minutes_per_week  0.567020
1           screen_time_hours -0.337552


Згідно з отриманими параметрами моделі, найбільш вагомим параметром виявився рівень стресу, чий від’ємний коефіцієнт демонструє найвищу значущість для класифікації. Зі збільшенням стресу ймовірність позитивного ментального стану стрімко знижується. 
Найбільший позитивний вплив на цільову змінну забезпечують якість сну та рівень продуктивності. Водночас модель фіксує антагоністичний вплив фізичних вправ та екранного часу: перший фактор сприяє благополуччю, другий — погіршує його. 
Також виявлено статистично значущу асоціацію між статусом студента та зниженням прогнозованого рівня благополуччя, що може бути наслідком кумулятивного стресу під час навчання